# Stage 8 — Base ASAG Textual Entailment & Propositional Relevance

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 8 — Textual Entailment / Relevance  

---

### Purpose
This notebook demonstrates the **Stage 8 Component** for the Base ASAG system.

In previous stages, we implemented:
- **Stage 3:** Linguistic Preprocessing (tokenization, lemmatization, POS tagging).
- **Stage 4:** Dependency Parsing (syntactic heads, grammatical dependency relations).
- **Stage 5:** Dependency Triples (raw arcs and canonical `(subject, predicate, object)` relations).
- **Stage 6:** Lexical and Semantic Relationships (4-level hierarchy: exact, lemma, POS guard, WordNet path).
- **Stage 7:** Dependency Triple Matching (role-preserving one-to-one alignment, continuous triple scores, and partitioning into matched, unmatched reference, and unmatched student relations).

**Stage 8 evaluates whether an aligned student proposition actually supports the reference proposition.**

Structural and lexical alignment (Stage 7) is necessary but insufficient to confirm that a student proposition supports a reference claim. For example:
- A student statement containing explicit negation (e.g. *"A primary key does not identify records"*) has high lexical and structural overlap with the reference answer, but **contradicts** it.
- A student statement with weak semantic support on a critical slot (e.g. general WordNet giving $0.125$ for `record` vs `row`, or weak verb alignment like `is associated with`) cannot be definitively asserted as an entailment without unwarranted assumptions; it must be classified as **uncertain**.

Stage 8 produces three interpretable proposition-level decisions:
1. **`ENTAILS`:** Sufficient structural alignment, compatible polarity, and all active slots satisfy minimum similarity thresholds.
2. **`CONTRADICTS`:** Explicit grammatical or lexical polarity clash on structurally aligned propositions.
3. **`UNCERTAIN`:** Insufficient semantic evidence or borderline alignment; decisions are never forced into a binary label.

> **CRITICAL RESEARCH BOUNDARIES:**  
> - Stage 8 evaluates **only aligned propositions** from Stage 7.
> - It does **not** classify unmatched student information (classification as correct additional, irrelevant, or incorrect belongs to the proposed extension in Stage 13).
> - It does **not** calculate student grades or answer-level scores (reserved for Stage 9).
> - It does **not** employ opaque large language models or sentence-level cosine similarities.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> Synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas for formatted output and the complete Base ASAG pipeline up to Stage 8:
- Stage 4: `DependencyParser`
- Stage 5: `DependencyTripleExtractor`
- Stage 7: `TripleMatcher`
- Stage 8: `PropositionalEntailmentAnalyzer`, `PolarityDetector`, `EntailmentResult`

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.dependency import DependencyParser, DependencyTripleExtractor
from base_asag.matching import TripleMatcher
from base_asag.entailment import PropositionalEntailmentAnalyzer, PolarityDetector, EntailmentResult

# Initialize pipeline components
parser = DependencyParser(model_name="en_core_web_sm")
extractor = DependencyTripleExtractor()
matcher = TripleMatcher(alignment_threshold=0.5)
analyzer = PropositionalEntailmentAnalyzer(
    entailment_threshold=0.70,
    min_slot_similarity=0.50,
    contradiction_threshold=0.50,
)

print("Initialized Stage 8 PropositionalEntailmentAnalyzer successfully.")
print(f"Entailment Threshold:    {analyzer.entailment_threshold}")
print(f"Min Slot Similarity:     {analyzer.min_slot_similarity}")
print(f"Contradiction Threshold: {analyzer.contradiction_threshold}")

Initialized Stage 8 PropositionalEntailmentAnalyzer successfully.
Entailment Threshold:    0.7
Min Slot Similarity:     0.5
Contradiction Threshold: 0.5


## 2. Demonstration 1: Identical Proposition (`ENTAILS`)

**Question:** *"What is the purpose of a primary key?"*  
- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key uniquely identifies each record in a table."*  

Both answers have identical surface forms, positive polarity, and exact slot matches.

In [2]:
ref_doc = parser.parse("A primary key uniquely identifies each record in a table.")
stu1_doc = parser.parse("A primary key uniquely identifies each record in a table.")

ref_triples = extractor.extract_document(ref_doc)
stu1_triples = extractor.extract_document(stu1_doc)

match_res1 = matcher.match(ref_triples, stu1_triples)
entail_res1 = analyzer.evaluate(match_res1.matched_relations[0], ref_doc.sentences[0], stu1_doc.sentences[0])

print(f"Proposition Decision: {entail_res1.relation}")
print(f"Confidence:           {entail_res1.confidence:.4f}")
print(f"Reference Polarity:   {entail_res1.reference_polarity}")
print(f"Student Polarity:     {entail_res1.student_polarity}")
print("\nDecision Reasons:")
for r in entail_res1.reasons:
    print(f"  - {r}")

Proposition Decision: ENTAILS
Confidence:           1.0000
Reference Polarity:   POSITIVE
Student Polarity:     POSITIVE

Decision Reasons:
  - Polarity is compatible (both propositions are POSITIVE).
  - Subject slot satisfies minimum similarity (1.0000 >= 0.50) via exact_match.
  - Predicate slot satisfies minimum similarity (1.0000 >= 0.50) via exact_match.
  - Object slot satisfies minimum similarity (1.0000 >= 0.50) via exact_match.
  - Composite triple alignment score (1.0000) meets entailment threshold (0.70).


## 3. Demonstration 2: Explicit Contradiction (`CONTRADICTS`)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key does not uniquely identify each record in a table."*  

In Stage 7, the underlying relation `(key, identify, record)` aligns with a high triple score ($1.0$).  
Stage 8 inspects the dependency parse, identifies the verbal negation particle `not` (`dep=neg`), and classifies the proposition as **`CONTRADICTS`**.

In [3]:
stu2_doc = parser.parse("A primary key does not uniquely identify each record in a table.")
stu2_triples = extractor.extract_document(stu2_doc)

match_res2 = matcher.match(ref_triples, stu2_triples)
entail_res2 = analyzer.evaluate(match_res2.matched_relations[0], ref_doc.sentences[0], stu2_doc.sentences[0])

print(f"Proposition Decision: {entail_res2.relation}")
print(f"Confidence:           {entail_res2.confidence:.4f}")
print(f"Reference Polarity:   {entail_res2.reference_polarity}")
print(f"Student Polarity:     {entail_res2.student_polarity}")
print(f"Negation Evidence:    {entail_res2.student_polarity_info.evidence}")
print("\nDecision Reasons:")
for r in entail_res2.reasons:
    print(f"  - {r}")

Proposition Decision: CONTRADICTS
Confidence:           1.0000
Reference Polarity:   POSITIVE
Student Polarity:     NEGATED
Negation Evidence:    Negation marker 'not' (dep=neg) attached to predicate 'identify'.

Decision Reasons:
  - Reference proposition expresses a positive claim.
  - Student proposition contains explicit negation (not).
  - Underlying proposition terms are sufficiently aligned (triple score: 1.0000 >= 0.50).
  - Explicit polarity contradiction on structurally corresponding dependency relations.


## 4. Demonstration 3: Paraphrased Domain Term (`UNCERTAIN`)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key uniquely identifies every row in a table."*  

In Stage 6, WordNet path similarity between domain terms `record` and `row` was found to be only **0.125** due to the absence of relational database synsets in general WordNet.  
In Stage 7, the triple score was $0.7083$.

Under conservative entailment rules (`min_slot_similarity = 0.50`), the object slot fails to satisfy the minimum certainty threshold. Rather than asserting false certainty, Stage 8 reports **`UNCERTAIN`** and preserves the exact slot evidence.

In [4]:
stu3_doc = parser.parse("A primary key uniquely identifies every row in a table.")
stu3_triples = extractor.extract_document(stu3_doc)

match_res3 = matcher.match(ref_triples, stu3_triples)
entail_res3 = analyzer.evaluate(match_res3.matched_relations[0], ref_doc.sentences[0], stu3_doc.sentences[0])

print(f"Proposition Decision: {entail_res3.relation}")
print(f"Confidence:           {entail_res3.confidence:.4f}")
print(f"Reference Polarity:   {entail_res3.reference_polarity}")
print(f"Student Polarity:     {entail_res3.student_polarity}")
print("\nDecision Reasons:")
for r in entail_res3.reasons:
    print(f"  - {r}")

Proposition Decision: UNCERTAIN
Confidence:           0.7083
Reference Polarity:   POSITIVE
Student Polarity:     POSITIVE

Decision Reasons:
  - Polarity is compatible (both propositions are POSITIVE), but semantic support is inconclusive.
  - Object similarity (0.1250) for 'record' <-> 'row' is below slot threshold (0.50).


### Research Transparency
> **RESEARCH NOTE:**  
> This illustrates why Stage 8 must support `UNCERTAIN`. When ontology resources fail to capture technical synonymy, forcing a binary entailment decision would distort research integrity. The limitation remains completely visible and measurable.

## 5. Demonstration 4: Weak Predicate Alignment (`UNCERTAIN`)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key is associated with records in a table."*  

The student uses a vague predicate (`is associated with` instead of `identifies`). The predicate similarity is weak, leading to an **`UNCERTAIN`** decision.

In [5]:
stu4_doc = parser.parse("A primary key is associated with records in a table.")
stu4_triples = extractor.extract_document(stu4_doc)

match_res4 = matcher.match(ref_triples, stu4_triples)

if match_res4.matched_count > 0:
    entail_res4 = analyzer.evaluate(match_res4.matched_relations[0], ref_doc.sentences[0], stu4_doc.sentences[0])
    print(f"Proposition Decision: {entail_res4.relation}")
    print(f"Confidence:           {entail_res4.confidence:.4f}")
    print("\nDecision Reasons:")
    for r in entail_res4.reasons:
        print(f"  - {r}")
else:
    print("Candidate pair did not meet alignment threshold in Stage 7.")

Proposition Decision: UNCERTAIN
Confidence:           0.5000

Decision Reasons:
  - Polarity is compatible (both propositions are POSITIVE), but semantic support is inconclusive.
  - Composite triple score (0.5000) is below entailment threshold (0.70).
  - Object similarity (0.0000) for 'record' <-> 'None' is below slot threshold (0.50).


## 6. Demonstration 5: Correct Answer + Additional Information (Student B)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key uniquely identifies each record in a table and cannot contain NULL values."*  

Stage 7 separated the answer into:
1. Matched relation: `(key, identifies, record)`
2. Unmatched student relation: `(key, contain, values)`

**Stage 8 Boundary Enforcement:**  
- The matched proposition is evaluated and classified as **`ENTAILS`**.  
- The unmatched student relation is **not classified** as correct, incorrect, or irrelevant. It is preserved strictly as unaligned information.

In [6]:
stu_b_doc = parser.parse(
    "A primary key uniquely identifies each record in a table and cannot contain NULL values."
)
stu_b_triples = extractor.extract_document(stu_b_doc)

match_res_b = matcher.match(ref_triples, stu_b_triples)

print(f"Stage 7 Matched Count:           {match_res_b.matched_count}")
print(f"Stage 7 Unmatched Student Count: {match_res_b.unmatched_student_count}\n")

# Evaluate entailment on matched propositions
entail_list_b = analyzer.evaluate_matching_result(match_res_b, ref_doc, stu_b_doc)

print("EVALUATED MATCHED PROPOSITIONS:")
for e in entail_list_b:
    print(f"  Relation:   {e.reference_relation.to_tuple()} <--> {e.student_relation.to_tuple()}")
    print(f"  Decision:   {e.relation} (Confidence: {e.confidence:.4f})")
    print(f"  Polarities: Ref={e.reference_polarity}, Student={e.student_polarity}")

print("\nUNMATCHED STUDENT INFORMATION (Untouched by Stage 8):")
for u in match_res_b.unmatched_student_relations:
    print(f"  Unmatched Relation: {u.to_tuple()} [Type: {u.relation_type}]")
    print(f"  Classification:     {getattr(u, 'classification', 'NONE (Reserved for Stage 13)')}")

Stage 7 Matched Count:           1
Stage 7 Unmatched Student Count: 1

EVALUATED MATCHED PROPOSITIONS:
  Relation:   ('key', 'identifies', 'record') <--> ('key', 'identifies', 'record')
  Decision:   ENTAILS (Confidence: 1.0000)
  Polarities: Ref=POSITIVE, Student=POSITIVE

UNMATCHED STUDENT INFORMATION (Untouched by Stage 8):
  Unmatched Relation: ('key', 'contain', 'values') [Type: SPO]
  Classification:     NONE (Reserved for Stage 13)


## 7. Demonstration 6: Correct Answer + Irrelevant Information (Student D)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student Answer:** *"A primary key uniquely identifies each record in a table. SQL was developed in the 1970s."*  

Similarly, the primary-key proposition evaluates to **`ENTAILS`**, while the SQL history relation `('SQL', 'developed', None)` remains unclassified in `unmatched_student_relations`.

In [7]:
stu_d_doc = parser.parse(
    "A primary key uniquely identifies each record in a table. SQL was developed in the 1970s."
)
stu_d_triples = extractor.extract_document(stu_d_doc)

match_res_d = matcher.match(ref_triples, stu_d_triples)
entail_list_d = analyzer.evaluate_matching_result(match_res_d, ref_doc, stu_d_doc)

print("EVALUATED MATCHED PROPOSITIONS:")
for e in entail_list_d:
    print(f"  Relation: {e.reference_relation.to_tuple()} <--> {e.student_relation.to_tuple()}")
    print(f"  Decision: {e.relation} (Confidence: {e.confidence:.4f})")

print("\nUNMATCHED STUDENT INFORMATION:")
for u in match_res_d.unmatched_student_relations:
    print(f"  Unmatched Relation: {u.to_tuple()} [Type: {u.relation_type}]")
    print(f"  Classification:     {getattr(u, 'classification', 'NONE (Reserved for Stage 13)')}")

EVALUATED MATCHED PROPOSITIONS:
  Relation: ('key', 'identifies', 'record') <--> ('key', 'identifies', 'record')
  Decision: ENTAILS (Confidence: 1.0000)

UNMATCHED STUDENT INFORMATION:
  Unmatched Relation: ('SQL', 'developed', None) [Type: PASSIVE_SP]
  Classification:     NONE (Reserved for Stage 13)


## 8. Summary Matrix Across All Demonstrations

In [8]:
summary_rows = [
    {
        "Scenario": "Identical Answer",
        "Student Text": stu1_doc.sentences[0].original_text,
        "Decision": entail_res1.relation,
        "Confidence": round(entail_res1.confidence, 4),
        "Polarities (Ref / Stu)": f"{entail_res1.reference_polarity} / {entail_res1.student_polarity}",
        "Primary Reason": entail_res1.reasons[0],
    },
    {
        "Scenario": "Explicit Negation",
        "Student Text": stu2_doc.sentences[0].original_text,
        "Decision": entail_res2.relation,
        "Confidence": round(entail_res2.confidence, 4),
        "Polarities (Ref / Stu)": f"{entail_res2.reference_polarity} / {entail_res2.student_polarity}",
        "Primary Reason": entail_res2.reasons[1],
    },
    {
        "Scenario": "Domain Term Paraphrase",
        "Student Text": stu3_doc.sentences[0].original_text,
        "Decision": entail_res3.relation,
        "Confidence": round(entail_res3.confidence, 4),
        "Polarities (Ref / Stu)": f"{entail_res3.reference_polarity} / {entail_res3.student_polarity}",
        "Primary Reason": entail_res3.reasons[-1],
    },
    {
        "Scenario": "Correct + Extra Info",
        "Student Text": stu_b_doc.sentences[0].original_text,
        "Decision": entail_list_b[0].relation,
        "Confidence": round(entail_list_b[0].confidence, 4),
        "Polarities (Ref / Stu)": f"{entail_list_b[0].reference_polarity} / {entail_list_b[0].student_polarity}",
        "Primary Reason": "Matched entails; 1 extra relation unclassified",
    },
    {
        "Scenario": "Correct + Irrelevant Info",
        "Student Text": stu_d_doc.sentences[0].original_text,
        "Decision": entail_list_d[0].relation,
        "Confidence": round(entail_list_d[0].confidence, 4),
        "Polarities (Ref / Stu)": f"{entail_list_d[0].reference_polarity} / {entail_list_d[0].student_polarity}",
        "Primary Reason": "Matched entails; 1 extra relation unclassified",
    },
]

pd.DataFrame(summary_rows)

,Scenario,Student Text,Decision,Confidence,Polarities (Ref / Stu),Primary Reason
0,Identical Answer,A primary key uniquely identifies each record ...,ENTAILS,1.0000,POSITIVE / POSITIVE,Polarity is compatible (both propositions are ...
1,Explicit Negation,A primary key does not uniquely identify each ...,CONTRADICTS,1.0000,POSITIVE / NEGATED,Student proposition contains explicit negation...
2,Domain Term Paraphrase,A primary key uniquely identifies every row in...,UNCERTAIN,0.7083,POSITIVE / POSITIVE,Object similarity (0.1250) for 'record' <-> 'r...
3,Correct + Extra Info,A primary key uniquely identifies each record ...,ENTAILS,1.0000,POSITIVE / POSITIVE,Matched entails; 1 extra relation unclassified
4,Correct + Irrelevant Info,A primary key uniquely identifies each record ...,ENTAILS,1.0000,POSITIVE / POSITIVE,Matched entails; 1 extra relation unclassified


## 9. Research Conclusions & Boundary Statement

### What Stage 8 Delivers
1. **Interpretable 3-Way Decisions:** Evaluates aligned relations into `ENTAILS`, `CONTRADICTS`, or `UNCERTAIN` based on transparent rules.
2. **Grammatical Polarity Detection:** Inspects verbal auxiliaries, negation particles, and nominal determiners in the dependency tree.
3. **Evidence-Based Explanations:** Every decision provides explicit human-readable reasons and retains slot-level evidence from Stages 6 and 7.
4. **Honest Uncertainty:** Faithfully identifies when lexical gaps (e.g. `record` vs `row`) lead to inconclusive support without forcing artificial assumptions.

### Strict Boundaries Maintained
- Stage 8 evaluates **only aligned propositions**.
- It does **not** classify unmatched student information (reserved for Stage 13).
- It does **not** compute a numerical answer grade or score (reserved for Stage 9).
- It does **not** use LLMs, deep sentence embeddings, or whole-answer cosine similarities.